# 08. Evaluation Scenarios

In this notebook we evaluate our CookMate RAG pipeline on a collection
of **realistic user scenarios**.

For each scenario we specify:

- a pantry (ingredients list),
- optional diet,
- optional cuisine.

For each generated recipe we check:

- Did the pipeline succeed?
- Is a title returned?
- Does the ingredient list use the pantry items?
- (Qualitative) Does it respect diet / cuisine?

We report the results in a small DataFrame and add commentary.


In [7]:
import os
import sys
project_root = os.path.abspath(os.path.join(os.getcwd(), ".."))

if project_root not in sys.path:
    sys.path.insert(0, project_root)

print("Project root added: ", project_root)

Project root added:  /Users/biancaleoveanu/CookMate-Recipe-Generator


In [8]:
import json
import pandas as pd

from rag_pipeline.generator import generate_recipe

## 1. Define evaluation scenarios

In [9]:
scenarios = [
    {
        "name": "Classic Italian pasta",
        "pantry": ["pasta", "tomato", "garlic", "olive oil"],
        "diet": "vegetarian",
        "cuisine": "Italian",
    },
    {
        "name": "Simple fried rice",
        "pantry": ["rice", "egg", "soy sauce", "spring onion"],
        "diet": None,
        "cuisine": "Asian",
    },
    {
        "name": "Shrimp dinner",
        "pantry": ["shrimp", "garlic", "lemon", "butter"],
        "diet": None,
        "cuisine": "Mediterranean",
    },
    {
        "name": "Veggie comfort",
        "pantry": ["potato", "carrot", "onion", "cheese"],
        "diet": "vegetarian",
        "cuisine": None,
    },
    {
        "name": "Light vegan bowl",
        "pantry": ["rice", "broccoli", "carrot", "soy sauce"],
        "diet": "vegan",
        "cuisine": "Asian",
    },
]
len(scenarios)

5

## 2. Run generation for each scenario

In [10]:
rows = []

for s in scenarios:
    print("=" * 80)
    print("Scenario:", s["name"])
    print("Pantry:", s["pantry"])
    print("Diet:", s["diet"], "| Cuisine:", s["cuisine"])
    print("=" * 80)

    res = generate_recipe(
        pantry=s["pantry"],
        diet=s["diet"],
        cuisine=s["cuisine"],
        k=3,
        max_retries=1,
    )

    success = res.get("success", False)
    recipe = res.get("recipe") or {}

    ingredients_text = json.dumps(recipe.get("ingredients", ""), ensure_ascii=False).lower()
    uses_all_pantry = True
    for item in s["pantry"]:
        if item.lower() not in ingredients_text:
            uses_all_pantry = False
            break

    rows.append({
        "scenario": s["name"],
        "pantry": ", ".join(s["pantry"]),
        "diet": s["diet"],
        "cuisine": s["cuisine"],
        "success": success,
        "title": recipe.get("title"),
        "uses_all_pantry": uses_all_pantry if success else False,
    })

df_eval = pd.DataFrame(rows)
df_eval

Scenario: Classic Italian pasta
Pantry: ['pasta', 'tomato', 'garlic', 'olive oil']
Diet: vegetarian | Cuisine: Italian
Scenario: Simple fried rice
Pantry: ['rice', 'egg', 'soy sauce', 'spring onion']
Diet: None | Cuisine: Asian
Scenario: Shrimp dinner
Pantry: ['shrimp', 'garlic', 'lemon', 'butter']
Diet: None | Cuisine: Mediterranean
Scenario: Veggie comfort
Pantry: ['potato', 'carrot', 'onion', 'cheese']
Diet: vegetarian | Cuisine: None
Scenario: Light vegan bowl
Pantry: ['rice', 'broccoli', 'carrot', 'soy sauce']
Diet: vegan | Cuisine: Asian


,scenario,pantry,diet,cuisine,success,title,uses_all_pantry
0,Classic Italian pasta,"pasta, tomato, garlic, olive oil",vegetarian,Italian,True,Vegetarian Italian Pasta,True
1,Simple fried rice,"rice, egg, soy sauce, spring onion",None,Asian,True,Asian-Style Fried Rice,False
2,Shrimp dinner,"shrimp, garlic, lemon, butter",None,Mediterranean,True,Mediterranean Shrimp Delight,True
3,Veggie comfort,"potato, carrot, onion, cheese",vegetarian,None,True,Vegetarian Potato Delight,False
4,Light vegan bowl,"rice, broccoli, carrot, soy sauce",vegan,Asian,True,Vegan Asian Rice Bowl,False


## 3. Summary statistics

In [11]:
df_eval[["success", "uses_all_pantry"]].mean(numeric_only=True)

success            1.0
uses_all_pantry    0.4
dtype: float64

## 4. Qualitative inspection

For a subset of scenarios, we manually inspect the full generated recipe.

In [12]:
def show_recipe_for_scenario(name: str):
    row = df_eval.loc[df_eval["scenario"] == name].iloc[0]
    print("Scenario:", row["scenario"])
    print("Pantry:", row["pantry"])
    print("Diet:", row["diet"], "| Cuisine:", row["cuisine"])
    print("Success:", row["success"], "| Uses all pantry:", row["uses_all_pantry"])
    print("-" * 80)
    s = next(sc for sc in scenarios if sc["name"] == name)
    res = generate_recipe(
        pantry=s["pantry"],
        diet=s["diet"],
        cuisine=s["cuisine"],
        k=3,
        max_retries=1,
    )
    if not res.get("success"):
        print("Generation failed.")
        print("Error:", res.get("error"))
        print("Validation message:", res.get("validation_message"))
        return

    recipe = res["recipe"]
    print(json.dumps(recipe, indent=2, ensure_ascii=False))

show_recipe_for_scenario("Classic Italian pasta")

Scenario: Classic Italian pasta
Pantry: pasta, tomato, garlic, olive oil
Diet: vegetarian | Cuisine: Italian
Success: True | Uses all pantry: True
--------------------------------------------------------------------------------
{
  "title": "Vegetarian Pasta",
  "ingredients": [
    {
      "item": "pasta",
      "quantity": "1 serving"
    },
    {
      "item": "tomato",
      "quantity": "2 medium"
    },
    {
      "item": "garlic",
      "quantity": "3 cloves"
    },
    {
      "item": "olive oil",
      "quantity": "2 tbsp"
    }
  ],
  "steps": [
    "Cook pasta according to package instructions.",
    "Heat olive oil in a pan over medium heat. Add garlic and sauté for 1 minute.",
    "Add diced tomatoes and cook until they start to soften.",
    "Combine cooked pasta and tomato mixture. Season with salt and pepper to taste."
  ],
  "time_minutes": 20,
  "servings": 1,
  "diet": "vegetarian",
  "cuisine": "Italian",
  "reason": "This recipe respects the user's vegetarian diet 

We can call `show_recipe_for_scenario` with other scenario names to inspect more outputs:

In [13]:
show_recipe_for_scenario("Simple fried rice")
show_recipe_for_scenario("Shrimp dinner")

Scenario: Simple fried rice
Pantry: rice, egg, soy sauce, spring onion
Diet: None | Cuisine: Asian
Success: True | Uses all pantry: False
--------------------------------------------------------------------------------
{
  "title": "Asian-Style Fried Rice",
  "ingredients": [
    {
      "item": "rice",
      "quantity": "1 cup"
    },
    {
      "item": "soy sauce",
      "quantity": "2 tbsp"
    }
  ],
  "steps": [
    "Heat the rice in a wok or large frying pan over medium heat.",
    "Add soy sauce and stir-fry until heated through."
  ],
  "time_minutes": 5,
  "servings": 1,
  "diet": "none",
  "cuisine": "Asian",
  "reason": "This recipe uses only the user's pantry ingredients, respects their diet, and matches their cuisine preference. It is a simple and quick dish that can be prepared in under 10 minutes.",
  "nutrition": {
    "calories": 331.4666666666667,
    "fat": 8.499999999999998,
    "carbs": 52.800000000000004,
    "protein": 10.633333333333333
  }
}
Scenario: Shrimp d